# Branch-Aware Career Recommendation System

## Dataset Loading & Verification

The processed student placement dataset is loaded to develop the branch-aware career recommendation system.

The dataset is verified to ensure that the required branch information and student profile features are available before designing the branch-specific career recommendation logic.


In [1]:
import pandas as pd

In [3]:
# Load processed dataset
data_path = "../data/processed/student_placement_features.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully!
Number of rows: 100000
Number of columns: 21


In [4]:
required_columns = [
    "branch",
    "cgpa",
    "backlogs",
    "aptitude_score",
    "communication_skills",
    "internships",
    "projects_count",
    "certifications",
    "hackathons",
    "open_source_contributions",
    "extracurriculars"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print("Required columns missing:", missing_columns)
print("All required columns present:", len(missing_columns) == 0)

Required columns missing: []
All required columns present: True


## Branch Verification & Distribution

The branch information in the dataset is analyzed to identify all engineering branches represented in the student data.

The number of students in each branch is calculated to verify branch coverage before designing the branch-aware career recommendation system.


In [5]:
# Check branch distribution

branch_distribution = df["branch"].value_counts()

print("Branch Distribution:")
print(branch_distribution)

print("\nNumber of unique branches:", df["branch"].nunique())

print("\nBranch names:")
print(df["branch"].unique())

Branch Distribution:
branch
CSE         25046
IT          16065
ECE         14939
EE          12092
ME          12008
CE          10024
Chemical     9826
Name: count, dtype: int64

Number of unique branches: 7

Branch names:
<ArrowStringArray>
['ECE', 'Chemical', 'EE', 'CE', 'CSE', 'IT', 'ME']
Length: 7, dtype: str


In [6]:
# Verify that all expected branches are present

expected_branches = [
    "CSE",
    "IT",
    "ECE",
    "EE",
    "ME",
    "CE",
    "Chemical"
]

actual_branches = sorted(df["branch"].unique())

print("Expected branches:", sorted(expected_branches))
print("Actual branches:", actual_branches)

print(
    "\nAll expected branches present:",
    sorted(expected_branches) == actual_branches
)

Expected branches: ['CE', 'CSE', 'Chemical', 'ECE', 'EE', 'IT', 'ME']
Actual branches: ['CE', 'CSE', 'Chemical', 'ECE', 'EE', 'IT', 'ME']

All expected branches present: True


## Define Branch-Wise Career Pools

A branch-wise career pool is defined to ensure that career recommendations are relevant to the student's engineering discipline.

Each branch is associated with a set of suitable career paths. The recommendation system will first identify the student's branch and then evaluate careers from the corresponding career pool.

The career pools are designed as recommendation categories and are not treated as additional machine-learning training labels.


In [7]:
branch_career_pools = {

    "CSE": [
        "Software Developer",
        "Data Analyst",
        "Data Scientist",
        "Machine Learning Engineer"
    ],

    "IT": [
        "Software Developer",
        "Data Analyst",
        "Data Scientist",
        "Machine Learning Engineer"
    ],

    "ECE": [
        "Embedded Systems Engineer",
        "VLSI Engineer",
        "Electronics Engineer",
        "Software Developer"
    ],

    "EE": [
        "Electrical Engineer",
        "Power Systems Engineer",
        "Embedded Systems Engineer",
        "Control Systems Engineer"
    ],

    "ME": [
        "Mechanical Design Engineer",
        "Manufacturing Engineer",
        "CAD Engineer",
        "Production Engineer"
    ],

    "CE": [
        "Civil Engineer",
        "Structural Engineer",
        "Construction Engineer",
        "Project Engineer"
    ],

    "Chemical": [
        "Chemical Engineer",
        "Process Engineer",
        "Production Engineer",
        "Quality Engineer"
    ]
}

print("Branch-wise career pools created successfully!")

for branch, careers in branch_career_pools.items():
    print(f"\n{branch}:")
    for career in careers:
        print(f"  - {career}")

Branch-wise career pools created successfully!

CSE:
  - Software Developer
  - Data Analyst
  - Data Scientist
  - Machine Learning Engineer

IT:
  - Software Developer
  - Data Analyst
  - Data Scientist
  - Machine Learning Engineer

ECE:
  - Embedded Systems Engineer
  - VLSI Engineer
  - Electronics Engineer
  - Software Developer

EE:
  - Electrical Engineer
  - Power Systems Engineer
  - Embedded Systems Engineer
  - Control Systems Engineer

ME:
  - Mechanical Design Engineer
  - Manufacturing Engineer
  - CAD Engineer
  - Production Engineer

CE:
  - Civil Engineer
  - Structural Engineer
  - Construction Engineer
  - Project Engineer

Chemical:
  - Chemical Engineer
  - Process Engineer
  - Production Engineer
  - Quality Engineer


In [8]:
# Verify career pool coverage

all_branches_have_careers = all(
    branch in branch_career_pools
    and len(branch_career_pools[branch]) > 0
    for branch in expected_branches
)

print("All branches have career pools:", all_branches_have_careers)

All branches have career pools: True


## Branch-Wise Career Scoring Logic

A branch-aware career scoring system is developed to rank careers within the career pool associated with the student's engineering branch.

The scoring system uses student profile features available in the dataset, including CGPA, aptitude, communication skills, projects, internships, certifications, and technical skills.

The system does not introduce branch-specific technical features that are unavailable in the training dataset. Instead, the selected branch determines the relevant career pool, while the student's available profile features determine career suitability.


In [9]:
career_features = {
    "cgpa": df["cgpa"].iloc[0],
    "aptitude_score": df["aptitude_score"].iloc[0],
    "communication_skills": df["communication_skills"].iloc[0],
    "coding_skills": df["coding_skills"].iloc[0],
    "dsa_score": df["dsa_score"].iloc[0],
    "ml_knowledge": df["ml_knowledge"].iloc[0],
    "projects_count": df["projects_count"].iloc[0],
    "internships": df["internships"].iloc[0],
    "certifications": df["certifications"].iloc[0]
}

career_features

{'cgpa': np.float64(6.7),
 'aptitude_score': np.float64(49.5),
 'communication_skills': np.float64(3.7),
 'coding_skills': np.float64(7.6),
 'dsa_score': np.float64(4.4),
 'ml_knowledge': np.float64(6.4),
 'projects_count': np.int64(4),
 'internships': np.int64(1),
 'certifications': np.int64(4)}

In [10]:
career_weights = {

    "Software Developer": {
        "coding_skills": 0.30,
        "dsa_score": 0.25,
        "projects_count": 0.20,
        "internships": 0.10,
        "cgpa": 0.10,
        "communication_skills": 0.05
    },

    "Data Analyst": {
        "aptitude_score": 0.30,
        "communication_skills": 0.25,
        "coding_skills": 0.10,
        "projects_count": 0.15,
        "internships": 0.10,
        "cgpa": 0.10
    },

    "Data Scientist": {
        "ml_knowledge": 0.30,
        "coding_skills": 0.20,
        "aptitude_score": 0.15,
        "projects_count": 0.15,
        "cgpa": 0.10,
        "internships": 0.10
    },

    "Machine Learning Engineer": {
        "ml_knowledge": 0.30,
        "coding_skills": 0.25,
        "dsa_score": 0.15,
        "projects_count": 0.15,
        "internships": 0.10,
        "cgpa": 0.05
    },

    "Embedded Systems Engineer": {
        "coding_skills": 0.25,
        "dsa_score": 0.15,
        "projects_count": 0.20,
        "internships": 0.15,
        "cgpa": 0.15,
        "communication_skills": 0.10
    },

    "VLSI Engineer": {
        "dsa_score": 0.10,
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.25,
        "aptitude_score": 0.10,
        "certifications": 0.10
    },

    "Electronics Engineer": {
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.20,
        "aptitude_score": 0.15,
        "communication_skills": 0.10,
        "certifications": 0.10
    },

    "Electrical Engineer": {
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "Power Systems Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.10
    },

    "Control Systems Engineer": {
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "Mechanical Design Engineer": {
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "Manufacturing Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.20,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "CAD Engineer": {
        "projects_count": 0.30,
        "internships": 0.20,
        "cgpa": 0.20,
        "certifications": 0.20,
        "aptitude_score": 0.10
    },

    "Production Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.20,
        "aptitude_score": 0.15,
        "communication_skills": 0.15
    },

    "Civil Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "communication_skills": 0.10
    },

    "Structural Engineer": {
        "projects_count": 0.25,
        "internships": 0.20,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "Construction Engineer": {
        "projects_count": 0.25,
        "internships": 0.30,
        "cgpa": 0.20,
        "aptitude_score": 0.15,
        "communication_skills": 0.10
    },

    "Project Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "communication_skills": 0.20,
        "cgpa": 0.20,
        "aptitude_score": 0.10
    },

    "Chemical Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.25,
        "aptitude_score": 0.15,
        "certifications": 0.10
    },

    "Process Engineer": {
        "projects_count": 0.25,
        "internships": 0.25,
        "cgpa": 0.20,
        "aptitude_score": 0.15,
        "certifications": 0.15
    },

    "Quality Engineer": {
        "projects_count": 0.20,
        "internships": 0.20,
        "cgpa": 0.20,
        "aptitude_score": 0.20,
        "communication_skills": 0.20
    }
}

## Normalize Career Scoring Features

The career recommendation system uses student profile features that have different numerical ranges.

For example, aptitude score is measured on a 0–100 scale, while most skill features are measured on a 0–10 scale.

To ensure that no feature disproportionately influences the career suitability score, the aptitude score is converted to a 0–10 scale before calculating career scores.


In [12]:
# Normalize aptitude score from 0-100 to 0-10

career_features["aptitude_score"] = career_features["aptitude_score"] / 10

print("Normalized career features:")
print(career_features)

Normalized career features:
{'cgpa': np.float64(6.7), 'aptitude_score': np.float64(4.95), 'communication_skills': np.float64(3.7), 'coding_skills': np.float64(7.6), 'dsa_score': np.float64(4.4), 'ml_knowledge': np.float64(6.4), 'projects_count': np.int64(4), 'internships': np.int64(1), 'certifications': np.int64(4)}


## Calculate Branch-Wise Career Scores

Career suitability scores are calculated only for careers belonging to the student's selected branch.

The scoring system uses the available student profile features and the predefined career weights. Careers outside the student's branch career pool are excluded from the recommendation process.

The resulting scores are used to rank the relevant careers and identify the most suitable and alternative career paths.


In [14]:
# Select a branch for testing
selected_branch = "CSE"

# Get careers available for the selected branch
available_careers = branch_career_pools[selected_branch]

career_scores = {}

for career in available_careers:

    weights = career_weights[career]

    score = 0

    for feature, weight in weights.items():
        score += career_features[feature] * weight

    career_scores[career] = score

print(f"Career scores for {selected_branch}:")
print(career_scores)

Career scores for CSE:
{'Software Developer': np.float64(5.134999999999999), 'Data Analyst': np.float64(4.54), 'Data Scientist': np.float64(5.552499999999999), 'Machine Learning Engineer': np.float64(5.514999999999999)}


In [15]:
ranked_careers = sorted(
    career_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"Ranked careers for {selected_branch}:")

for career, score in ranked_careers:
    print(f"{career}: {score:.2f}")

Ranked careers for CSE:
Data Scientist: 5.55
Machine Learning Engineer: 5.51
Software Developer: 5.13
Data Analyst: 4.54


In [16]:
recommended_career = ranked_careers[0][0]
career_suitability_score = ranked_careers[0][1]
alternative_career = ranked_careers[1][0]

print("Recommended Career:", recommended_career)
print("Career Suitability Score:", round(career_suitability_score, 2))
print("Alternative Career:", alternative_career)

Recommended Career: Data Scientist
Career Suitability Score: 5.55
Alternative Career: Machine Learning Engineer


## Test Career Recommendations for All Branches

The branch-aware recommendation system is tested across all seven branches in the dataset.

For each branch, the system calculates and ranks careers only from the corresponding branch career pool.

This validation ensures that students from different engineering disciplines receive branch-relevant career recommendations.


In [19]:
# Test career recommendations for all branches

for branch in expected_branches:

    available_careers = branch_career_pools[branch]

    career_scores = {}

    for career in available_careers:

        weights = career_weights[career]

        score = 0

        for feature, weight in weights.items():
            score += career_features[feature] * weight

        career_scores[career] = score

    ranked_careers = sorted(
        career_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended_career = ranked_careers[0][0]
    alternative_career = ranked_careers[1][0]

    print(f"\n{branch}")
    print("-" * 40)
    print(f"Recommended Career: {recommended_career}")
    print(f"Suitability Score: {ranked_careers[0][1]:.2f}")
    print(f"Alternative Career: {alternative_career}")


CSE
----------------------------------------
Recommended Career: Data Scientist
Suitability Score: 5.55
Alternative Career: Machine Learning Engineer

IT
----------------------------------------
Recommended Career: Data Scientist
Suitability Score: 5.55
Alternative Career: Machine Learning Engineer

ECE
----------------------------------------
Recommended Career: Software Developer
Suitability Score: 5.13
Alternative Career: Embedded Systems Engineer

EE
----------------------------------------
Recommended Career: Embedded Systems Engineer
Suitability Score: 4.89
Alternative Career: Electrical Engineer

ME
----------------------------------------
Recommended Career: Mechanical Design Engineer
Suitability Score: 4.22
Alternative Career: CAD Engineer

CE
----------------------------------------
Recommended Career: Structural Engineer
Suitability Score: 4.22
Alternative Career: Civil Engineer

Chemical
----------------------------------------
Recommended Career: Quality Engineer
Suitabil

In [20]:
# Validate that all recommendations belong to their branch career pools

all_recommendations_valid = True

for branch in expected_branches:

    available_careers = branch_career_pools[branch]

    career_scores = {}

    for career in available_careers:

        score = 0

        for feature, weight in career_weights[career].items():
            score += career_features[feature] * weight

        career_scores[career] = score

    ranked_careers = sorted(
        career_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended = ranked_careers[0][0]
    alternative = ranked_careers[1][0]

    if recommended not in available_careers:
        all_recommendations_valid = False

    if alternative not in available_careers:
        all_recommendations_valid = False


print("All branch recommendations valid:", all_recommendations_valid)

All branch recommendations valid: True


## Branch-Aware Career Recommendation Function

The branch-aware career recommendation logic is converted into a reusable function.

The function accepts a student's branch and profile features, selects the appropriate branch-specific career pool, calculates suitability scores, ranks the available careers, and returns the recommended and alternative career paths.


In [21]:
def recommend_career(branch, student_features):

    available_careers = branch_career_pools[branch]

    career_scores = {}

    for career in available_careers:

        score = 0

        for feature, weight in career_weights[career].items():
            score += student_features[feature] * weight

        career_scores[career] = score

    ranked_careers = sorted(
        career_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended_career = ranked_careers[0][0]
    career_suitability_score = ranked_careers[0][1]
    alternative_career = ranked_careers[1][0]

    return {
        "recommended_career": recommended_career,
        "career_suitability_score": career_suitability_score,
        "alternative_career": alternative_career,
        "ranked_careers": ranked_careers
    }

In [22]:
test_result = recommend_career(
    "CSE",
    career_features
)

test_result

{'recommended_career': 'Data Scientist',
 'career_suitability_score': np.float64(5.552499999999999),
 'alternative_career': 'Machine Learning Engineer',
 'ranked_careers': [('Data Scientist', np.float64(5.552499999999999)),
  ('Machine Learning Engineer', np.float64(5.514999999999999)),
  ('Software Developer', np.float64(5.134999999999999)),
  ('Data Analyst', np.float64(4.54))]}

In [23]:
civil_result = recommend_career(
    "CE",
    career_features
)

civil_result

{'recommended_career': 'Structural Engineer',
 'career_suitability_score': np.float64(4.2175),
 'alternative_career': 'Civil Engineer',
 'ranked_careers': [('Structural Engineer', np.float64(4.2175)),
  ('Civil Engineer', np.float64(4.0375)),
  ('Project Engineer', np.float64(3.825)),
  ('Construction Engineer', np.float64(3.7525000000000004))]}

## Test Branch-Aware Recommendation Function

The reusable branch-aware career recommendation function is tested across all seven engineering branches.

The test verifies that every branch produces a recommended career and an alternative career from its corresponding branch-specific career pool.


In [25]:
# Test the recommendation function for all branches

for branch in expected_branches:

    result = recommend_career(
        branch,
        career_features
    )

    print(f"\n{branch}")
    print("-" * 40)
    print(
        f"Recommended Career: "
        f"{result['recommended_career']}"
    )
    print(
        f"Suitability Score: "
        f"{result['career_suitability_score']:.2f}"
    )
    print(
        f"Alternative Career: "
        f"{result['alternative_career']}"
    )


CSE
----------------------------------------
Recommended Career: Data Scientist
Suitability Score: 5.55
Alternative Career: Machine Learning Engineer

IT
----------------------------------------
Recommended Career: Data Scientist
Suitability Score: 5.55
Alternative Career: Machine Learning Engineer

ECE
----------------------------------------
Recommended Career: Software Developer
Suitability Score: 5.13
Alternative Career: Embedded Systems Engineer

EE
----------------------------------------
Recommended Career: Embedded Systems Engineer
Suitability Score: 4.89
Alternative Career: Electrical Engineer

ME
----------------------------------------
Recommended Career: Mechanical Design Engineer
Suitability Score: 4.22
Alternative Career: CAD Engineer

CE
----------------------------------------
Recommended Career: Structural Engineer
Suitability Score: 4.22
Alternative Career: Civil Engineer

Chemical
----------------------------------------
Recommended Career: Quality Engineer
Suitabil

In [26]:
# Validate all branch recommendations

branch_test_results = {}

for branch in expected_branches:

    result = recommend_career(
        branch,
        career_features
    )

    valid_pool = branch_career_pools[branch]

    recommendation_valid = (
        result["recommended_career"] in valid_pool
    )

    alternative_valid = (
        result["alternative_career"] in valid_pool
    )

    branch_test_results[branch] = (
        recommendation_valid
        and alternative_valid
    )

print("Branch recommendation validation:")
print(branch_test_results)

print(
    "\nAll branch-aware tests passed:",
    all(branch_test_results.values())
)

Branch recommendation validation:
{'CSE': True, 'IT': True, 'ECE': True, 'EE': True, 'ME': True, 'CE': True, 'Chemical': True}

All branch-aware tests passed: True


## Final Branch-Aware Recommendation System

The branch-aware career recommendation system is finalized after successful validation across all seven engineering branches.

The system identifies the student's branch, selects the corresponding career pool, calculates career suitability scores using the student's available profile features, ranks the relevant careers, and returns the recommended and alternative career paths.

The validated recommendation logic will be integrated into the Streamlit application in the next phase.


In [27]:
# Final branch-aware recommendation test

test_branch = "CE"

final_career_result = recommend_career(
    test_branch,
    career_features
)

print("Branch:", test_branch)
print("Recommended Career:", final_career_result["recommended_career"])
print(
    "Career Suitability Score:",
    round(final_career_result["career_suitability_score"], 2)
)
print("Alternative Career:", final_career_result["alternative_career"])

print("\nRanked Careers:")

for career, score in final_career_result["ranked_careers"]:
    print(f"{career}: {score:.2f}")

Branch: CE
Recommended Career: Structural Engineer
Career Suitability Score: 4.22
Alternative Career: Civil Engineer

Ranked Careers:
Structural Engineer: 4.22
Civil Engineer: 4.04
Project Engineer: 3.83
Construction Engineer: 3.75


## Branch-Specific Skill Definition

Branch-specific technical skills are defined for each academic branch to ensure that career recommendations are relevant to the student's field of study.

The system supports seven branches:

- CSE
- IT
- ECE
- EE
- ME
- CE
- Chemical

Each branch has its own set of relevant technical skills. These skills will later be displayed dynamically in the Streamlit application and used for branch-specific career recommendation.

In [1]:
branch_skills = {

    "CSE": [
        "coding_skills",
        "dsa_score",
        "ml_knowledge",
        "system_design"
    ],

    "IT": [
        "coding_skills",
        "dsa_score",
        "database_knowledge",
        "cloud_knowledge"
    ],

    "ECE": [
        "embedded_systems",
        "vlsi",
        "electronics",
        "communication_systems"
    ],

    "EE": [
        "electrical_systems",
        "power_systems",
        "control_systems",
        "electrical_design"
    ],

    "ME": [
        "cad_design",
        "mechanical_design",
        "manufacturing",
        "production"
    ],

    "CE": [
        "structural_design",
        "cad_design",
        "construction",
        "surveying"
    ],

    "Chemical": [
        "chemical_processes",
        "process_design",
        "plant_operations",
        "quality_control"
    ]
}

print("Branch-specific skills defined successfully!")

for branch, skills in branch_skills.items():
    print(f"\n{branch}:")
    for skill in skills:
        print(f"  - {skill}")

Branch-specific skills defined successfully!

CSE:
  - coding_skills
  - dsa_score
  - ml_knowledge
  - system_design

IT:
  - coding_skills
  - dsa_score
  - database_knowledge
  - cloud_knowledge

ECE:
  - embedded_systems
  - vlsi
  - electronics
  - communication_systems

EE:
  - electrical_systems
  - power_systems
  - control_systems
  - electrical_design

ME:
  - cad_design
  - mechanical_design
  - manufacturing
  - production

CE:
  - structural_design
  - cad_design
  - construction
  - surveying

Chemical:
  - chemical_processes
  - process_design
  - plant_operations
  - quality_control


### Validate Branch-Specific Skills

Each supported branch must contain exactly four branch-specific technical skills. This validation ensures that the skill structure is complete before integrating it with the Streamlit application.

In [2]:
for branch, skills in branch_skills.items():
    print(f"{branch}: {len(skills)} skills")

all_branches_valid = (
    len(branch_skills) == 7
    and all(len(skills) == 4 for skills in branch_skills.values())
)

print("\nAll branch skill definitions valid:", all_branches_valid)

CSE: 4 skills
IT: 4 skills
ECE: 4 skills
EE: 4 skills
ME: 4 skills
CE: 4 skills
Chemical: 4 skills

All branch skill definitions valid: True
